# FraudBench Experiment Runner (Colab)

This notebook runs **all** experiments on Colab.

## Original Experiments (already complete)
- **Cell 7** — GPU batch: Neural model experiments (CAPGD, adversarial training, input validation, epsilon sweeps)
- **Cell 8** — CPU batch: Tree model experiments (CAPGD baseline, input validation, HopSkipJump, Square Attack)

## MVP Supplement Experiments (NEW)
- **Cell 9** — Ensemble defence: 4 datasets × 3 seeds × 2 attacks = **24 runs** (GPU)
- **Cell 10** — Z-threshold sweep: 4 neural + 4 tree = **8 runs** (GPU + CPU, seed 42 only)
- **Cell 11** — Post-experiment analysis: statistical tests + figures

## Housekeeping
- **Cell 12** — Save results to Google Drive
- **Cell 13** — Session cleanup

**Tip:** If you only need to run the MVP supplement experiments, skip Cells 7–8 and go straight to Cells 9–10.

**Workflow:**
1. Mount Google Drive for data and results persistence
2. Clone/update the repo
3. Install dependencies
4. Link datasets from Google Drive
5. Run experiments (Cells 7–10)
6. Run analysis (Cell 11)
7. Save results to Google Drive (Cell 12)

In [ ]:
# Cell 1: Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "models", "logs"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
    print(f"  {DRIVE_ROOT}/{subdir}/")

print("\nGoogle Drive mounted and directories ready.")

In [ ]:
# Cell 3: Clone or update repo
import os
import shutil

REPO_URL = "https://github.com/iHaydenzZ/FraudBench.git"
REPO_DIR = "/content/FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Repo exists. Pulling latest changes...")
    os.chdir(REPO_DIR)
    !git pull
else:
    # Always start from /content so git clone has a valid CWD
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
        print("Removed stale directory (not a git repo).")
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"\nWorking directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
# Colab does not have uv, so we use pip install -e . (editable mode)
!pip install -e . -q 2>&1 | tail -5

# Upgrade numba to work with Colab's NumPy 2.x (ART imports brendel_bethge -> numba)
!pip install "numba>=0.61" -q 2>&1 | tail -3

In [ ]:
# Cell 5: Symlink dataset directories from Google Drive
# The loader expects data at <project_root>/datasets/<DatasetName>/
# We symlink individual dirs (not the whole datasets/ folder,
# because it also contains Python source files like loader.py)
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        files = os.listdir(src)
        print(f"  Linked: {dataset_dir}/ ({len(files)} files)")
    else:
        print(f"  NOT FOUND on Drive: {dataset_dir}/ -- upload to {src}")

print("\nDataset symlinks ready.")

In [ ]:
# Cell 6: Run a single experiment
# Change CONFIG to run different experiments.
# Available configs: ls configs/

CONFIG = "configs/ccfd.yaml"

!python -m runner.run --config {CONFIG}

In [ ]:
# Cell 7: GPU batch -- All Neural model experiments x 3 seeds
# These use PyTorch for model training and CAPGD gradient computation.
# Runtime: T4 GPU (~1.96 units/hr)
# Total: 16 configs x 3 seeds = 48 experiments
import os
import subprocess
import time
from tqdm.notebook import tqdm

SEEDS = [42, 123, 456]

gpu_configs = [
    # --- CCFD (neural) ---
    "configs/ccfd.yaml",                   # neural baseline + CAPGD
    "configs/ccfd_adv_train.yaml",         # neural + adversarial training
    "configs/ccfd_input_val.yaml",         # neural + input validation
    "configs/ccfd_eps_sweep.yaml",         # neural + CAPGD epsilon sweep

    # --- IEEE-CIS (neural) ---
    "configs/ieee_cis_neural.yaml",        # neural baseline + CAPGD
    "configs/ieee_cis_adv_train.yaml",     # neural + adversarial training
    "configs/ieee_cis_input_val.yaml",     # neural + input validation
    "configs/ieee_cis_eps_sweep.yaml",     # neural + CAPGD epsilon sweep

    # --- LCLD (neural) ---
    "configs/lcld.yaml",                   # neural baseline + CAPGD
    "configs/lcld_adv_train.yaml",         # neural + adversarial training
    "configs/lcld_input_val.yaml",         # neural + input validation
    "configs/lcld_eps_sweep.yaml",         # neural + CAPGD epsilon sweep

    # --- Sparkov (neural) ---
    "configs/sparkov.yaml",                # neural baseline + CAPGD
    "configs/sparkov_adv_train.yaml",      # neural + adversarial training
    "configs/sparkov_input_val.yaml",      # neural + input validation
    "configs/sparkov_eps_sweep.yaml",      # neural + CAPGD epsilon sweep
]

def _short_name(path):
    return os.path.splitext(os.path.basename(path))[0]

experiments = [(config, seed) for config in gpu_configs for seed in SEEDS]
failed = []

for config, seed in tqdm(experiments, desc="GPU experiments", unit="exp"):
    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config, "--seed", str(seed)],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"\nFAILED ({elapsed:.1f}s): {_short_name(config)} s{seed}")
        print(result.stderr[-500:] if result.stderr else "No error output")
        failed.append(f"{config} --seed {seed}")

total = len(experiments)
print(f"\nGPU batch done: {total - len(failed)}/{total} succeeded.")
if failed:
    print(f"Failed ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")
print("\nRun Cell 8 to save results to Drive.")

In [ ]:
# Cell 8: CPU batch -- All Tree model experiments x 3 seeds
# XGBoost + CAPGD baselines, input validation, HopSkipJump, Square Attack
# These are CPU-only -- no GPU needed. Use a CPU runtime to save compute units.
# Total: 16 configs x 3 seeds = 48 experiments
import os
import subprocess
import time
from tqdm.notebook import tqdm

SEEDS = [42, 123, 456]

cpu_configs = [
    # --- CCFD (tree) ---
    "configs/ccfd_tree.yaml",              # tree baseline + CAPGD
    "configs/ccfd_tree_input_val.yaml",    # tree + input validation
    "configs/ccfd_tree_hsj.yaml",          # tree + HopSkipJump
    "configs/ccfd_tree_square.yaml",       # tree + Square Attack

    # --- IEEE-CIS (tree) ---
    "configs/ieee_cis.yaml",              # tree baseline + CAPGD
    "configs/ieee_cis_tree_input_val.yaml", # tree + input validation
    "configs/ieee_cis_tree_hsj.yaml",      # tree + HopSkipJump
    "configs/ieee_cis_tree_square.yaml",   # tree + Square Attack

    # --- LCLD (tree) ---
    "configs/lcld_tree.yaml",              # tree baseline + CAPGD
    "configs/lcld_tree_input_val.yaml",    # tree + input validation
    "configs/lcld_tree_hsj.yaml",          # tree + HopSkipJump
    "configs/lcld_tree_square.yaml",       # tree + Square Attack

    # --- Sparkov (tree) ---
    "configs/sparkov_tree.yaml",           # tree baseline + CAPGD
    "configs/sparkov_tree_input_val.yaml", # tree + input validation
    "configs/sparkov_tree_hsj.yaml",       # tree + HopSkipJump
    "configs/sparkov_tree_square.yaml",    # tree + Square Attack
]

def _short_name(path):
    return os.path.splitext(os.path.basename(path))[0]

experiments = [(config, seed) for config in cpu_configs for seed in SEEDS]
failed = []

for config, seed in tqdm(experiments, desc="CPU experiments", unit="exp"):
    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config, "--seed", str(seed)],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"\nFAILED ({elapsed:.1f}s): {_short_name(config)} s{seed}")
        print(result.stderr[-500:] if result.stderr else "No error output")
        failed.append(f"{config} --seed {seed}")

total = len(experiments)
print(f"\nCPU batch done: {total - len(failed)}/{total} succeeded.")
if failed:
    print(f"Failed ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")
print("\nRun Cell 9 to save results to Drive.")

---

# MVP Supplement Experiments

The cells below run the **new** experiments needed to complete the MVP:
- **Ensemble defence** — adds a 3rd defence method (LR + XGBoost + MLP soft-voting)
- **Z-threshold sweep** — deepens the input validation analysis with z={5.0, 10.0}

In [ ]:
# Cell 9: Ensemble Defence — 4 datasets x 3 seeds x 2 attacks = 24 runs
# All runs need GPU (MLP component in ensemble uses CUDA).
# CAPGD runs are fast (gradient-based). Square runs are slower (query-based).
# Sequential execution (1 at a time) to avoid GPU OOM on T4.
import os
import subprocess
import time
from tqdm.notebook import tqdm

SEEDS = [42, 123, 456]

ensemble_configs = [
    # --- CAPGD (white-box via MLP gradients, fast) ---
    "configs/ccfd_ensemble.yaml",
    "configs/ieee_cis_ensemble.yaml",
    "configs/lcld_ensemble.yaml",
    "configs/sparkov_ensemble.yaml",

    # --- Square Attack (black-box, slower but sample_frac=0.1) ---
    "configs/ccfd_ensemble_square.yaml",
    "configs/ieee_cis_ensemble_square.yaml",
    "configs/lcld_ensemble_square.yaml",
    "configs/sparkov_ensemble_square.yaml",
]

def _short_name(path):
    return os.path.splitext(os.path.basename(path))[0]

experiments = [(config, seed) for config in ensemble_configs for seed in SEEDS]
failed = []

print(f"Running {len(experiments)} ensemble experiments sequentially...")
print(f"Configs: {len(ensemble_configs)} | Seeds: {SEEDS}\n")

for config, seed in tqdm(experiments, desc="Ensemble experiments", unit="exp"):
    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config, "--seed", str(seed)],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    status = "OK" if result.returncode == 0 else "FAIL"
    print(f"  [{status}] {_short_name(config)} seed={seed} ({elapsed:.1f}s)")

    if result.returncode != 0:
        print(f"    Error: {result.stderr[-500:]}" if result.stderr else "    No error output")
        failed.append(f"{config} --seed {seed}")

total = len(experiments)
print(f"\nEnsemble batch done: {total - len(failed)}/{total} succeeded.")
if failed:
    print(f"Failed ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")

In [ ]:
# Cell 10: Z-Threshold Sweep — 8 runs (seed 42 only)
# Tests input validation with z_threshold = {5.0, 10.0} on CCFD and Sparkov.
# Neural configs need GPU; tree configs are CPU-only (CAPGD is a no-op on trees).
# All 8 runs are lightweight — single seed, no heavy attacks.
import os
import subprocess
import time
from tqdm.notebook import tqdm

SEEDS = [42]  # Single seed for sweep analysis

z_sweep_configs = [
    # --- Neural (GPU: training + CAPGD gradients) ---
    "configs/ccfd_input_val_z5.yaml",
    "configs/ccfd_input_val_z10.yaml",
    "configs/sparkov_input_val_z5.yaml",
    "configs/sparkov_input_val_z10.yaml",

    # --- Tree (CPU: XGBoost + CAPGD no-op) ---
    "configs/ccfd_tree_input_val_z5.yaml",
    "configs/ccfd_tree_input_val_z10.yaml",
    "configs/sparkov_tree_input_val_z5.yaml",
    "configs/sparkov_tree_input_val_z10.yaml",
]

def _short_name(path):
    return os.path.splitext(os.path.basename(path))[0]

experiments = [(config, seed) for config in z_sweep_configs for seed in SEEDS]
failed = []

print(f"Running {len(experiments)} z-threshold sweep experiments...")
print(f"Configs: {len(z_sweep_configs)} | Seeds: {SEEDS}\n")

for config, seed in tqdm(experiments, desc="Z-threshold sweep", unit="exp"):
    start = time.time()
    result = subprocess.run(
        ["python", "-m", "runner.run", "--config", config, "--seed", str(seed)],
        capture_output=True, text=True
    )
    elapsed = time.time() - start

    status = "OK" if result.returncode == 0 else "FAIL"
    print(f"  [{status}] {_short_name(config)} seed={seed} ({elapsed:.1f}s)")

    if result.returncode != 0:
        print(f"    Error: {result.stderr[-500:]}" if result.stderr else "    No error output")
        failed.append(f"{config} --seed {seed}")

total = len(experiments)
print(f"\nZ-threshold sweep done: {total - len(failed)}/{total} succeeded.")
if failed:
    print(f"Failed ({len(failed)}):")
    for f in failed:
        print(f"  - {f}")

In [ ]:
# Cell 11: Post-Experiment Analysis
# Run this AFTER Cells 9-10 complete. Regenerates statistical tests and figures
# using the updated registry (now including ensemble + z-sweep data).
import subprocess

print("=" * 60)
print("1/3  Statistical significance tests")
print("=" * 60)
result = subprocess.run(
    ["python", "scripts/statistical_tests.py", "--registry", "results/registry_clean.csv"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print(f"ERROR: {result.stderr[-500:]}")

print("\n" + "=" * 60)
print("2/3  Adversarial training trade-off analysis")
print("=" * 60)
result = subprocess.run(
    ["python", "scripts/analyse_adv_training.py"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print(f"ERROR: {result.stderr[-500:]}")

print("\n" + "=" * 60)
print("3/3  Generate all figures")
print("=" * 60)
result = subprocess.run(
    ["python", "scripts/generate_figures.py", "--registry", "results/registry_clean.csv"],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print(f"ERROR: {result.stderr[-500:]}")

print("\n" + "=" * 60)
print("Done! Check results/figures/ for outputs.")
print("Run Cell 12 to save everything to Google Drive.")
print("=" * 60)

In [ ]:
# Cell 12: Copy results to Google Drive for persistence
import shutil
import glob
import os

RESULTS_SRC = "/content/FraudBench/results"
RESULTS_DST = "/content/drive/MyDrive/FraudBench/results"

for f in glob.glob(os.path.join(RESULTS_SRC, "*")):
    if os.path.isfile(f):
        dst = os.path.join(RESULTS_DST, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f"Saved: {os.path.basename(f)}")

# Also copy figures if they exist
figures_src = os.path.join(RESULTS_SRC, "figures")
figures_dst = os.path.join(RESULTS_DST, "figures")
if os.path.isdir(figures_src):
    if os.path.exists(figures_dst):
        shutil.rmtree(figures_dst)
    shutil.copytree(figures_src, figures_dst)
    print("Saved: figures/ directory")

print(f"\nAll results backed up to {RESULTS_DST}")

In [ ]:
# Cell 13: Session cleanup
# IMPORTANT: Run this when done to stop consuming compute units!
# First make sure results are saved (run Cell 12 first!)

print("Make sure you have saved results to Google Drive (Cell 12) before disconnecting!")
print("To disconnect: Runtime > Disconnect and delete runtime")
print("Or uncomment the line below:")
# from google.colab import runtime; runtime.unassign()